In [6]:
#!/usr/bin/env python3
"""
Golden Dataset Generator for RAG Evaluation using RAGAS
Reads markdown files and generates synthetic question-answer pairs.
"""

import json
import os
from pathlib import Path
from typing import Any

import nest_asyncio
import pandas as pd
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from openai import OpenAI
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.testset import TestsetGenerator

nest_asyncio.apply()

In [7]:
def read_markdown_files(data_dir: str, max_chars: int = 10000) -> list[dict[str, Any]]:
    """Read all markdown files from the specified directory and subdirectories."""
    data_path = Path(data_dir)
    documents = []

    for md_file in sorted(data_path.rglob("*.md")):
        try:
            with open(md_file, encoding="utf-8") as f:
                content = f.read()

            if content.strip():
                truncated = content[:max_chars] if len(content) > max_chars else content
                documents.append(
                    {"path": str(md_file.relative_to(data_path)), "content": truncated}
                )
        except Exception as e:
            print(f"Error reading {md_file}: {e}")

    return documents


def create_langchain_documents(docs: list[dict[str, Any]]) -> list[Document]:
    """Convert documents to LangChain Document format."""
    return [Document(page_content=doc["content"], metadata={"source": doc["path"]}) for doc in docs]


docs = read_markdown_files("../data/interim/docling")
print(f"Read {len(docs)} markdown files")
documents = create_langchain_documents(docs)
print(f"Created {len(documents)} LangChain documents")

Read 1395 markdown files
Created 1395 LangChain documents


In [19]:
api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_API_BASE")

if not api_key or not base_url:
    raise ValueError("OPENAI_API_KEY and OPENAI_API_BASE environment variables required.")

llm = ChatOpenAI(
    model="yandex/gpt-pro-5.1",
    api_key=api_key,
    base_url=base_url,
    max_tokens=4096,
    max_completion_tokens=4096,
)
openai_client = OpenAI(api_key=api_key, base_url=base_url)

evaluator_llm = LangchainLLMWrapper(llm)

embedding_model = HuggingFaceEmbeddings(
    model="sentence-transformers/distiluse-base-multilingual-cased-v1"
)

testset_generator = TestsetGenerator(llm=evaluator_llm, embedding_model=embedding_model)

print("TestsetGenerator initialized successfully")

/var/folders/z7/1s9cnr916tv6tyfwwc07jtn00000gn/T/ipykernel_88799/2787592843.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)


TestsetGenerator initialized successfully


In [21]:
testset = testset_generator.generate_with_langchain_docs(
    documents=documents[:1], testset_size=10, raise_exceptions=True
)

print(f"Generated {len(testset.samples)} test samples")

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/3 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

Generated 11 test samples


In [32]:
testset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Как узнать условия по кредитке Платинум и поль...,[Подписки и сервисы](/bank/help/#q6) <!-- imag...,Чтобы узнать условия по кредитной карте «Плати...,"Анна, владелец банковской карты",MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
1,Как узнать условия по кредитной карте «Платину...,[Подписки и сервисы](/bank/help/#q6) <!-- imag...,Чтобы узнать условия по кредитной карте «Плати...,"Анна, владелец банковской карты",PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,Как проверить и оплатить штрафи гибдд?,[рассрочку в магазине](/bank/help/refinance) <...,"Вы можете проверить, оплатить или оспорить штр...","Анна, владелец банковской карты",MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
3,Какие привилегии даёт сервис Premium и как к н...,[рассрочку в магазине](/bank/help/refinance) <...,"Сервис Premium предоставляет привилегии, однак...","Анна, владелец банковской карты",WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
4,Где найти информацию о подписках и сервисах ба...,[# \n\nТ-Помощь. Банк\n\nБаза знаний для клиен...,Информацию о подписках и сервисах можно найти ...,"Анна, владелец банковской карты",WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
5,Что можно найти в разделе «Т-Помощь» банка?,[# \n\nТ-Помощь. Банк\n\nБаза знаний для клиен...,В разделе «Т-Помощь» банка можно найти базу зн...,"Анна, владелец банковской карты",PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
6,Как подключить подписку Pro и какие преимущест...,[<1-hop>\n\nПодписки и сервисы](/bank/help/#q6...,"Чтобы подключить подписку Pro, нужно воспользо...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,Как падиключить саписску Pro? И какие приимуще...,[<1-hop>\n\nПодписки и сервисы](/bank/help/#q6...,"Чтобы подключить подписку Pro, нужно воспользо...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,Какие преимущества даёт подписка Pro от Т-Банк...,[<1-hop>\n\nПодписки и сервисы](/bank/help/#q6...,Подписка Pro от Т-Банка предоставляет следующи...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,Какие преимущества даёт подписка Pro в Т-Банке...,[<1-hop>\n\nПодписки и сервисы](/bank/help/#q6...,Подписка Pro в Т-Банке предоставляет следующие...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [31]:
samples = []

for item in testset.samples:
    sample = item.eval_sample
    print(sample.__dict__.keys())
    samples.append(
        {
            "question": sample.user_input,
            "answer": sample.reference,
            "context": sample.reference_context,
            "ground_truth": sample.reference,
        }
    )

output_path = Path("../data/interim/golden_dataset.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(samples, f, ensure_ascii=False, indent=2)

print(f"Saved {len(samples)} samples to {output_path}")

                                           user_input  \
0   Как узнать условия по кредитке Платинум и поль...   
1   Как узнать условия по кредитной карте «Платину...   
2              Как проверить и оплатить штрафи гибдд?   
3   Какие привилегии даёт сервис Premium и как к н...   
4   Где найти информацию о подписках и сервисах ба...   
5         Что можно найти в разделе «Т-Помощь» банка?   
6   Как подключить подписку Pro и какие преимущест...   
7   Как падиключить саписску Pro? И какие приимуще...   
8   Какие преимущества даёт подписка Pro от Т-Банк...   
9   Какие преимущества даёт подписка Pro в Т-Банке...   
10  Как подключить подписку Pro и какие преимущест...   

                                   reference_contexts  \
0   [Подписки и сервисы](/bank/help/#q6) <!-- imag...   
1   [Подписки и сервисы](/bank/help/#q6) <!-- imag...   
2   [рассрочку в магазине](/bank/help/refinance) <...   
3   [рассрочку в магазине](/bank/help/refinance) <...   
4   [# \n\nТ-Помощь. Банк\n\nБ

AttributeError: 'SingleTurnSample' object has no attribute 'reference_context'

In [ ]:
print(f"\n=== Generated {len(samples)} Golden Dataset Samples ===\n")
for i, sample in enumerate(samples, 1):
    print(f"--- Sample {i} ---")
    print(f"Q: {sample['question']}")
    print(f"A: {sample['answer']}")
    print()

In [ ]:
df = pd.DataFrame(samples)
df[["question", "answer"]]